<a href="https://colab.research.google.com/github/MuhammadSahlanHabibi/Impelmentasi-Seminar-Make-Sense-of-Data-with-analysis-and-AI/blob/main/Retail_Catchment_Area_Analysis_using_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# TAHAP 1: PERSIAPAN LINGKUNGAN KERJA (Jalankan di Google Colab)
# ==============================================================================
# Hapus tanda pagar pada baris di bawah ini jika library belum terpasang di Colab
# !pip install geopandas shapely folium matplotlib

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from shapely.geometry import Point
import matplotlib.pyplot as plt

print("Library berhasil dimuat. Memulai pemrosesan GIS...\n")

# ==============================================================================
# TAHAP 2: PEMBUATAN DATASET SPASIAL (GERAI & POPULASI)
# ==============================================================================
# 1. Data Lokasi Gerai Ritel (Koordinat Asli Bengkulu)
data_gerai = {
    'id_gerai': ['G-01', 'G-02', 'G-03'],
    'nama': ['Gerai Suprapto (Pusat Kota)', 'Gerai Unib Belakang', 'Gerai Pantai Panjang (BenMall)'],
    'lat': [-3.7963, -3.7580, -3.8119397], # Kordinat asli
    'lon': [102.2618, 102.2710, 102.2678883]
}
df_gerai = pd.DataFrame(data_gerai)
gdf_gerai = gpd.GeoDataFrame(
    df_gerai,
    geometry=gpd.points_from_xy(df_gerai.lon, df_gerai.lat),
    crs="EPSG:4326"
)

# 2. Simulasi Data Pemukiman Warga (Demand Points)
# Membuat 300 titik acak di sekitar kordinat Bengkulu untuk mensimulasikan kepadatan
np.random.seed(42)
lon_random = np.random.uniform(102.2700, 102.3100, 400) # Murni daratan timur
lat_random = np.random.uniform(-3.8300, -3.7500, 400)

gdf_populasi = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(lon_random, lat_random),
    crs="EPSG:4326"
)
gdf_populasi['id_rumah'] = [f'R-{i}' for i in range(400)]

# ==============================================================================
# TAHAP 3: GEOPROCESSING & ANALISIS SPASIAL TINGKAT LANJUT
# ==============================================================================
# Mengubah Proyeksi ke EPSG:32748 (UTM Zone 48S - Sumatera) agar kalkulasi menggunakan satuan Meter
gdf_gerai_utm = gdf_gerai.to_crs(epsg=32748)
gdf_pop_utm = gdf_populasi.to_crs(epsg=32748)

# Membuat Area Jangkauan (Buffer) radius 1.5 KM (1500 Meter)
gdf_gerai_utm['geometry'] = gdf_gerai_utm.geometry.buffer(1500)

# Melakukan SPATIAL JOIN: Menghitung rumah mana saja yang masuk ke dalam radius tiap gerai
# Operasi ini ekuivalen dengan fitur "Intersect" pada ArcGIS/QGIS
analisis_pasar = gpd.sjoin(gdf_pop_utm, gdf_gerai_utm, how='inner', predicate='intersects')

# Agregasi data: Hitung total rumah per gerai
potensi_pasar = analisis_pasar.groupby('nama')['id_rumah'].count().reset_index()
potensi_pasar.columns = ['Nama Gerai', 'Jumlah Rumah Terjangkau']

print("=== HASIL ANALISIS POTENSI PASAR ===")
print(potensi_pasar)
print("\n")

# Kembalikan geometri buffer ke EPSG:4326 untuk keperluan visualisasi Web Map
gdf_buffer_gps = gdf_gerai_utm.to_crs(epsg=4326)

# ==============================================================================
# TAHAP 4: CLOUD INTEGRATION (Export ke GeoJSON)
# ==============================================================================
# Menyimpan hasil buffer sebagai GeoJSON agar siap diintegrasikan ke backend web (Laravel/Node)
output_filename = "catchment_area_bengkulu.geojson"
gdf_buffer_gps.to_file(output_filename, driver="GeoJSON")
print(f"Data spasial berhasil diekspor ke: {output_filename}")

# ==============================================================================
# TAHAP 5: INTERACTIVE WEB MAPPING (Visulisasi ala Cloud GIS Dashboard)
# ==============================================================================
# Titik tengah peta
peta_bengkulu = folium.Map(
    location=[-3.7850, 102.2600],
    zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri'
)

# Layer 1: Titik Pemukiman (Bintik abu-abu)
for idx, row in gdf_populasi.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color='gray',
        fill=True,
        fill_opacity=0.5
    ).add_to(peta_bengkulu)

# Layer 2: Catchment Area / Buffer 1.5 KM (Poligon Biru)
folium.GeoJson(
    gdf_buffer_gps,
    name="Area Jangkauan (1.5 KM)",
    style_function=lambda feature: {
        'fillColor': '#3186cc',
        'color': '#3186cc',
        'weight': 2,
        'fillOpacity': 0.2
    },
    tooltip=folium.GeoJsonTooltip(fields=['nama'], aliases=['Nama Gerai:'])
).add_to(peta_bengkulu)

# Layer 3: Titik Gerai Utama (Marker Merah)
for idx, row in gdf_gerai.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"<b>{row['nama']}</b>",
        icon=folium.Icon(color='red', icon='shopping-cart')
    ).add_to(peta_bengkulu)

# Tambahkan kontrol layer agar pengguna bisa menghidup/matikan layer
folium.LayerControl().add_to(peta_bengkulu)

# Tampilkan Peta
peta_bengkulu

Library berhasil dimuat. Memulai pemrosesan GIS...

=== HASIL ANALISIS POTENSI PASAR ===
                       Nama Gerai  Jumlah Rumah Terjangkau
0  Gerai Pantai Panjang (BenMall)                       29
1     Gerai Suprapto (Pusat Kota)                       16
2             Gerai Unib Belakang                       31


Data spasial berhasil diekspor ke: catchment_area_bengkulu.geojson
